# 05 双模型雏鸟候选筛查

本模块比较成鸟/雏鸟训练集构成，并在相同三秒窗口上联合分析两套模型。主定义固定为：chick score ≥ 0.80 且 adult score < 0.80；both-high 窗口作为 general storm petrel 对照。扫描录音若来自任一模型的训练来源，会从论文主结果中排除；全量结果仅作敏感性对照。

In [8]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "project_config.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_config import WORKSPACE_DIR

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE_DIR / "matplotlib_cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

def require_columns(frame, required, name):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

def save_figure(fig, stem):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.close(fig)

def write_latex(frame, name, **kwargs):
    (OUTPUT_DIR / name).write_text(frame.to_latex(index=False, **kwargs), encoding="utf-8")

from scipy.stats import rankdata, wilcoxon
from project_config import CHICK_SCREENING_DIR

ADULT_THRESHOLD = 0.80
CHICK_THRESHOLD = 0.80
THRESHOLD_GRID = np.round(np.arange(0.50, 0.951, 0.05), 2)
BOOTSTRAP_ITERATIONS = 5000
RANDOM_SEED = 2021
OUTPUT_DIR = WORKSPACE_DIR / "paper_analysis" / "05_chick_screening"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ADULT_LABELS_PATH = CHICK_SCREENING_DIR / "adult_clips" / "labels.csv"
CHICK_LABELS_PATH = CHICK_SCREENING_DIR / "chick_clips" / "labels.csv"
SCORES_PATH = CHICK_SCREENING_DIR / "dual_scores.csv"
FEATURES_PATH = CHICK_SCREENING_DIR / "analysis" / "window_features.csv"
for path in [ADULT_LABELS_PATH, CHICK_LABELS_PATH, SCORES_PATH, FEATURES_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /home/zenith/Projects/strom_petrels/workspace/paper_analysis/05_chick_screening


## 两套训练数据集的构成

类别、负类亚型与来源录音数分别汇总。这里描述训练材料，不把片段数当作独立的生物学样本数。

In [9]:
adult_labels = pd.read_csv(ADULT_LABELS_PATH)
chick_labels = pd.read_csv(CHICK_LABELS_PATH)
for name, frame in [("adult", adult_labels), ("chick", chick_labels)]:
    require_columns(frame, {"clip_id", "label", "target_label", "negative_type", "source_file"}, f"{name} labels")

training_summary_rows = []
for model, frame in [("adult", adult_labels), ("chick", chick_labels)]:
    for (label, negative_type), group in frame.assign(negative_type=frame.negative_type.fillna("not_applicable")).groupby(["label", "negative_type"]):
        training_summary_rows.append({"model": model, "label": label, "negative_type": negative_type,
                                      "clips": len(group), "source_recordings": group.source_file.nunique()})
training_composition = pd.DataFrame(training_summary_rows)
training_totals = pd.DataFrame([
    {"model": model, "clips": len(frame), "positive_clips": int((frame.target_label == 1).sum()),
     "negative_clips": int((frame.target_label == 0).sum()), "source_recordings": frame.source_file.nunique()}
    for model, frame in [("adult", adult_labels), ("chick", chick_labels)]
])
training_composition.to_csv(OUTPUT_DIR / "training_composition.csv", index=False)
training_totals.to_csv(OUTPUT_DIR / "training_totals.csv", index=False)
write_latex(training_totals, "training_totals.tex")
display(training_totals)
display(training_composition)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
training_totals.set_index("model")[["positive_clips", "negative_clips"]].plot.bar(ax=axes[0], color=["#d95f02", "#6c757d"])
axes[0].set(title="Training class composition", xlabel="Model", ylabel="Clips")
axes[0].tick_params(axis="x", rotation=0)
neg_plot = training_composition[training_composition.negative_type != "not_applicable"].pivot(index="model", columns="negative_type", values="clips").fillna(0)
neg_plot.plot.bar(ax=axes[1])
axes[1].set(title="Training negative subtypes", xlabel="Model", ylabel="Clips")
axes[1].tick_params(axis="x", rotation=0)
save_figure(fig, "training_composition")

,model,clips,positive_clips,negative_clips,source_recordings
0,adult,324,162,162,3
1,chick,72,36,36,6


,model,label,negative_type,clips,source_recordings
0,adult,not_storm_petrel,environment_noise,80,3
1,adult,not_storm_petrel,other_bird,82,3
2,adult,storm_petrel,not_applicable,162,3
3,chick,not_storm_petrel,environment_noise,18,1
4,chick,not_storm_petrel,other_bird,18,6
5,chick,storm_petrel,not_applicable,36,6


## 排除训练来源重叠并定义四象限

“重叠”指扫描录音的文件名出现在成鸟或雏鸟任一训练集的 `source_file` 中，并非两套训练集彼此共享来源。当前数据应识别 9 条此类扫描录音。论文主结果排除它们，全量结果保留为敏感性对照。

In [10]:
scores = pd.read_csv(SCORES_PATH, usecols=["recording", "start_seconds", "end_seconds", "prob_adult", "prob_chick", "delta_prob"])
require_columns(scores, {"recording", "start_seconds", "prob_adult", "prob_chick"}, "dual scores")
if not scores[["prob_adult", "prob_chick"]].apply(lambda column: column.between(0, 1, inclusive="both").all()).all():
    raise AssertionError("Dual-model sigmoid scores must be in [0, 1]")

training_sources = set(adult_labels.source_file.astype(str)) | set(chick_labels.source_file.astype(str))
scores["recording_name"] = scores.recording.astype(str).str.replace(r"^.*[\\/]", "", regex=True)
scanned_sources = set(scores.recording_name)
overlap_sources = sorted(training_sources & scanned_sources)
if len(overlap_sources) != 9:
    raise AssertionError(f"Expected 9 scan/training source overlaps; found {len(overlap_sources)}: {overlap_sources}")
scores["training_source_overlap"] = scores.recording_name.isin(overlap_sources)

def add_quadrant(frame, adult_threshold=ADULT_THRESHOLD, chick_threshold=CHICK_THRESHOLD):
    adult_high = frame.prob_adult >= adult_threshold
    chick_high = frame.prob_chick >= chick_threshold
    return np.select([chick_high & ~adult_high, chick_high & adult_high, ~chick_high & adult_high],
                     ["chick_candidate", "general_storm_petrel", "adult_biased"], default="background")

scores["quadrant"] = add_quadrant(scores)
main_scores = scores[~scores.training_source_overlap].copy()
pd.DataFrame({"excluded_training_source": overlap_sources}).to_csv(OUTPUT_DIR / "excluded_training_sources.csv", index=False)
print(f"All recordings: {scores.recording.nunique()}; main-analysis recordings: {main_scores.recording.nunique()}; excluded: {len(overlap_sources)}")
display(pd.DataFrame({"excluded_training_source": overlap_sources}))

All recordings: 69; main-analysis recordings: 60; excluded: 9


,excluded_training_source
0,011_210522_184504.flac
1,011_210531_221151.flac
2,042_210803_211120.flac
3,064_210531_221134.flac
4,064_210803_211227.flac
5,064_210808_223734.flac
6,064_210815_222530.flac
7,111_210808_223734.flac
8,112_210803_211227.flac


In [11]:
def quadrant_summary(frame, scope):
    result = frame.groupby("quadrant").agg(window_count=("recording", "size"), recording_count=("recording", "nunique"),
                                             mean_adult_score=("prob_adult", "mean"), mean_chick_score=("prob_chick", "mean")).reset_index()
    result.insert(0, "scope", scope)
    return result

quadrants = pd.concat([quadrant_summary(main_scores, "main_excluding_training_sources"),
                       quadrant_summary(scores, "sensitivity_all_recordings")], ignore_index=True)
quadrants.to_csv(OUTPUT_DIR / "quadrant_summary.csv", index=False)
write_latex(quadrants[quadrants.scope == "main_excluding_training_sources"], "quadrant_summary.tex", float_format="%.4f")
display(quadrants)

grid_rows = []
for adult_threshold in THRESHOLD_GRID:
    adult_low = main_scores.prob_adult.to_numpy() < adult_threshold
    for chick_threshold in THRESHOLD_GRID:
        selected = adult_low & (main_scores.prob_chick.to_numpy() >= chick_threshold)
        grid_rows.append({"adult_threshold": adult_threshold, "chick_threshold": chick_threshold,
                          "candidate_windows": int(selected.sum()),
                          "candidate_recordings": main_scores.loc[selected, "recording"].nunique()})
threshold_grid = pd.DataFrame(grid_rows)
threshold_grid.to_csv(OUTPUT_DIR / "threshold_sensitivity_grid.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, value, title in [(axes[0], "candidate_windows", "Chick-candidate windows"),
                         (axes[1], "candidate_recordings", "Recordings with candidates")]:
    matrix = threshold_grid.pivot(index="chick_threshold", columns="adult_threshold", values=value).sort_index(ascending=False)
    image = ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set(xticks=np.arange(len(matrix.columns)), xticklabels=[f"{x:.2f}" for x in matrix.columns],
           yticks=np.arange(len(matrix.index)), yticklabels=[f"{x:.2f}" for x in matrix.index],
           xlabel="Adult threshold (candidate requires score below)", ylabel="Chick threshold", title=title)
    ax.tick_params(axis="x", rotation=45)
    fig.colorbar(image, ax=ax, fraction=0.046)
save_figure(fig, "threshold_sensitivity_heatmaps")

,scope,quadrant,window_count,recording_count,mean_adult_score,mean_chick_score
0,main_excluding_training_sources,adult_biased,1056,15,0.872972,0.727093
1,main_excluding_training_sources,background,1022774,60,0.008772,0.098924
2,main_excluding_training_sources,chick_candidate,1693,21,0.636126,0.846380
3,main_excluding_training_sources,general_storm_petrel,5550,18,0.946361,0.913293
4,sensitivity_all_recordings,adult_biased,1836,19,0.868791,0.732998
5,sensitivity_all_recordings,background,1164702,69,0.009794,0.100037
6,sensitivity_all_recordings,chick_candidate,2761,25,0.639929,0.851797
7,sensitivity_all_recordings,general_storm_petrel,14818,22,0.957807,0.925060


## 双模型分数、录音级候选率与日期趋势

散点图为了可读性固定随机抽样窗口；所有计数与统计使用完整主分析数据。录音级候选率以该录音全部三秒窗口为分母。

In [12]:
plot_sample = main_scores.sample(n=min(50000, len(main_scores)), random_state=RANDOM_SEED)
colors = {"chick_candidate": "#d62728", "general_storm_petrel": "#2ca02c",
          "adult_biased": "#1f77b4", "background": "#aaaaaa"}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for quadrant, group in plot_sample.groupby("quadrant"):
    axes[0].scatter(group.prob_adult, group.prob_chick, s=4, alpha=0.25, c=colors[quadrant], label=quadrant)
axes[0].axvline(ADULT_THRESHOLD, color="black", linestyle="--")
axes[0].axhline(CHICK_THRESHOLD, color="black", linestyle="--")
axes[0].set(title="Dual-model score quadrants (sampled for display)", xlabel="Adult model score", ylabel="Chick model score", xlim=(0, 1), ylim=(0, 1))
axes[0].legend(markerscale=3, fontsize=7)
axes[1].hist(main_scores.prob_adult, bins=60, density=True, histtype="step", linewidth=1.5, label="Adult model")
axes[1].hist(main_scores.prob_chick, bins=60, density=True, histtype="step", linewidth=1.5, label="Chick model")
axes[1].set(title="Dual-model score distributions", xlabel="Uncalibrated sigmoid model score", ylabel="Density", yscale="log")
axes[1].legend()
save_figure(fig, "dual_model_scores")

recording_summary = main_scores.groupby("recording").agg(
    total_windows=("recording", "size"),
    chick_candidate_count=("quadrant", lambda values: int((values == "chick_candidate").sum())),
    general_storm_petrel_count=("quadrant", lambda values: int((values == "general_storm_petrel").sum())),
).reset_index()
recording_summary["chick_candidate_rate"] = recording_summary.chick_candidate_count / recording_summary.total_windows
recording_summary["general_storm_petrel_rate"] = recording_summary.general_storm_petrel_count / recording_summary.total_windows
recording_summary["recording_date"] = pd.to_datetime(
    recording_summary.recording.astype(str).str.extract(r"_(\d{6})_")[0], format="%y%m%d", errors="coerce"
)
recording_summary.to_csv(OUTPUT_DIR / "recording_candidate_rates.csv", index=False)

ordered = recording_summary.sort_values("chick_candidate_rate", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(np.arange(len(ordered)), ordered.chick_candidate_rate, color="#d62728")
axes[0].set(title="Recording-level chick-candidate rate", xlabel="Source recordings (ranked)", ylabel="Candidate windows / all windows")
date_summary = recording_summary.groupby("recording_date").agg(candidate_rate=("chick_candidate_rate", "median"), recordings=("recording", "size")).reset_index()
axes[1].plot(date_summary.recording_date, date_summary.candidate_rate, marker="o", color="#d62728")
axes[1].set(title="Date trend in recording-level candidate rate", xlabel="Recording date", ylabel="Median candidate rate")
save_figure(fig, "recording_rates_and_date")

## 录音内配对声学比较

只纳入同一录音中同时存在 chick candidate 与 general storm petrel 的录音。每个声学特征先在“录音 × 类别”内取中位数，然后进行双侧 Wilcoxon signed-rank、配对 rank-biserial effect size 与 5,000 次录音级 paired bootstrap 95% CI。窗口级汇总仅作描述，不用于推断。主结果排除训练来源，全量结果作为敏感性对照。

In [13]:
features = pd.read_csv(FEATURES_PATH)
feature_names = [name for name in ["rms", "dbfs", "peak_frequency_hz", "spectral_centroid_hz", "spectral_bandwidth_hz",
                                            "rolloff_85_hz", "spectral_flatness", "spectral_entropy", "zero_crossing_rate"] if name in features.columns]
require_columns(features, {"recording", "quadrant", *feature_names}, "window features")
features["recording_name"] = features.recording.astype(str).str.replace(r"^.*[\\/]", "", regex=True)
features["training_source_overlap"] = features.recording_name.isin(overlap_sources)

def rank_biserial_paired(differences):
    nonzero = differences[np.abs(differences) > 0]
    if not len(nonzero):
        return 0.0
    ranks = rankdata(np.abs(nonzero))
    return (ranks[nonzero > 0].sum() - ranks[nonzero < 0].sum()) / ranks.sum()

def paired_effects(frame, scope, seed):
    rng = np.random.default_rng(seed)
    rows = []
    for feature in feature_names:
        paired = (frame[frame.quadrant.isin(["chick_candidate", "general_storm_petrel"])]
                  .groupby(["recording", "quadrant"])[feature].median().unstack().dropna())
        differences = (paired["chick_candidate"] - paired["general_storm_petrel"]).to_numpy(float)
        if len(differences) < 2:
            rows.append({"scope": scope, "feature": feature, "paired_recordings": len(differences)})
            continue
        test = wilcoxon(differences, alternative="two-sided", zero_method="wilcox")
        draws = np.median(rng.choice(differences, size=(BOOTSTRAP_ITERATIONS, len(differences)), replace=True), axis=1)
        low, high = np.quantile(draws, [0.025, 0.975])
        rows.append({"scope": scope, "feature": feature, "paired_recordings": len(differences),
                     "median_paired_difference_chick_minus_general": np.median(differences),
                     "bootstrap_ci95_low": low, "bootstrap_ci95_high": high,
                     "wilcoxon_statistic": test.statistic, "p_value": test.pvalue,
                     "paired_rank_biserial": rank_biserial_paired(differences)})
    return pd.DataFrame(rows)

paired_results = pd.concat([
    paired_effects(features[~features.training_source_overlap], "main_excluding_training_sources", RANDOM_SEED),
    paired_effects(features, "sensitivity_all_recordings", RANDOM_SEED + 1),
], ignore_index=True)

def bh_fdr(group):
    values = group.p_value.to_numpy(float)
    valid = np.flatnonzero(np.isfinite(values))
    adjusted = np.full(len(values), np.nan)
    if len(valid):
        order = valid[np.argsort(values[valid])]
        ranked = values[order] * len(order) / np.arange(1, len(order) + 1)
        adjusted[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    group = group.copy()
    group["p_value_bh_fdr"] = adjusted
    return group

paired_results = pd.concat(
    [bh_fdr(group) for _, group in paired_results.groupby("scope", sort=False)],
    ignore_index=True,
)
paired_results.to_csv(OUTPUT_DIR / "paired_recording_feature_effects.csv", index=False)
write_latex(paired_results[paired_results.scope == "main_excluding_training_sources"], "paired_recording_feature_effects.tex", float_format="%.4g")

window_descriptive = features.groupby("quadrant")[feature_names].agg(["count", "median"]).reset_index()
window_descriptive.columns = ["_".join([str(part) for part in column if part]) if isinstance(column, tuple) else column for column in window_descriptive.columns]
window_descriptive.insert(0, "statistical_level", "window_descriptive_only")
window_descriptive.to_csv(OUTPUT_DIR / "window_level_descriptive_features.csv", index=False)
display(paired_results)

,scope,feature,paired_recordings,median_paired_difference_chick_minus_general,bootstrap_ci95_low,bootstrap_ci95_high,wilcoxon_statistic,p_value,paired_rank_biserial,p_value_bh_fdr
0,main_excluding_training_sources,rms,17,-0.000047,-0.000637,0.000416,66.0,0.644135,-0.137255,0.644135
1,main_excluding_training_sources,dbfs,17,-0.409907,-1.661912,1.003029,59.0,0.430679,-0.228758,0.553731
2,main_excluding_training_sources,peak_frequency_hz,17,0.000000,0.000000,0.000000,6.0,0.345448,0.428571,0.518171
3,main_excluding_training_sources,spectral_centroid_hz,17,-45.874844,-128.799024,24.667587,48.0,0.190094,-0.372549,0.427711
4,main_excluding_training_sources,spectral_bandwidth_hz,17,128.207304,22.459296,417.944906,22.0,0.007904,0.712418,0.071136
5,main_excluding_training_sources,rolloff_85_hz,17,-20.739658,-285.633993,90.082621,62.0,0.517090,-0.189542,0.581726
6,main_excluding_training_sources,spectral_flatness,17,0.000053,-0.000003,0.000170,30.0,0.026672,0.607843,0.120026
7,main_excluding_training_sources,spectral_entropy,17,0.015758,-0.005108,0.036045,54.0,0.306046,0.294118,0.518171
8,main_excluding_training_sources,zero_crossing_rate,17,-0.001581,-0.006302,0.000145,39.0,0.079681,-0.490196,0.239044
9,sensitivity_all_recordings,rms,21,-0.000047,-0.000237,0.000332,107.0,0.785365,-0.073593,0.785365


In [14]:
main_effects = paired_results[paired_results.scope == "main_excluding_training_sources"].dropna(subset=["median_paired_difference_chick_minus_general"])
fig, ax = plt.subplots(figsize=(9, 5.5))
positions = np.arange(len(main_effects))
lower = main_effects.median_paired_difference_chick_minus_general - main_effects.bootstrap_ci95_low
upper = main_effects.bootstrap_ci95_high - main_effects.median_paired_difference_chick_minus_general
ax.errorbar(main_effects.median_paired_difference_chick_minus_general, positions,
            xerr=np.vstack([lower, upper]), fmt="o", color="#4c78a8", capsize=3)
ax.axvline(0, color="gray", linestyle="--")
ax.set(yticks=positions, yticklabels=main_effects.feature,
       title="Recording-paired acoustic differences", xlabel="Median difference (chick candidate − general storm petrel)")
save_figure(fig, "paired_acoustic_differences")

print("Main screening results exclude all 9 scan/training source overlaps; all-recording results are sensitivity only.")
print("Model scores are uncalibrated and window-level acoustic summaries are descriptive only.")

Main screening results exclude all 9 scan/training source overlaps; all-recording results are sensitivity only.
Model scores are uncalibrated and window-level acoustic summaries are descriptive only.
